# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Dataset metadata object (see fields below for details)
metadata = dataset.metadata
print("Dataset Name:", metadata.name)
print("Description:", metadata.description)


## 2. Data Overview
Review available record sets, fields, and their IDs.

We can enumerate the record sets in this dataset by their `@id` as defined in the Croissant schema. For each record set, we show available fields and their `@id`s.

In [ ]:
# List all record sets and their fields using the Croissant metadata
record_sets = list(dataset.record_sets)

if not record_sets:
    print("No record sets defined in schema. Attempting to enumerate from 'distribution'.")
    # As a fallback, use the distribution to list file objects (if any)
    for dist in getattr(metadata, 'distribution', []):
        print(json.dumps(dist, indent=2))
else:
    for rs in record_sets:
        print(f"Record set: {rs['@id']}")
        # Fields if available per record set
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        for field in fields:
            print(f"  Field: {field['@id']}  (name: {field.get('name', '-')})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Use the record set and field `@id`s from the overview.

In [ ]:
from pprint import pprint

# If record sets are defined, use their IDs.
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

# If no croissant record sets, try loading from distribution (file-based dataset)
if not record_set_ids:
    print("No record sets found in the schema. Listing distributions (likely file objects):")
    record_set_ids = [dist['@id'] for dist in getattr(metadata, 'distribution', [])]
    pprint(record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    try:
        print(f"Extracting records from record set or distribution: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame for {record_set_id}:")
            print(df.head())
        else:
            print(f"No records extracted for {record_set_id}.")
    except Exception as e:
        print(f"Error loading {record_set_id}: {e}")

# Pick the first available DataFrame for demonstration
record_set_key = record_set_ids[0] if record_set_ids else None
if record_set_key and record_set_key in dataframes:
    print(f"Available columns in {record_set_key}:")
    print(dataframes[record_set_key].columns.tolist())
    display(dataframes[record_set_key].head())
else:
    print("No DataFrame successfully loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes.

All field/column access is done via their `@id` strings as shown above.

In [ ]:
import numpy as np

"""Determine a numeric field, such as 'coverage' or 'MW' (molecular weight) by column name, typically present in protein datasets.
Field identifiers might be like 'coverage', 'MW', 'Peptides', 'pI', etc.
"""

if record_set_key and record_set_key in dataframes:
    df = dataframes[record_set_key]
    # Heuristics: look for numeric-valued columns
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if not numeric_fields:
        # Try converting possible columns
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except Exception:
                continue
        numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

    if numeric_fields:
        numeric_field = numeric_fields[0]
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.3f}:")
        print(filtered_df.head())
        # Normalize
        normalized_col = f"{numeric_field}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, normalized_col]].head())
        # Try grouping by a possibly categorical column
        group_field = None
        candidate_group_fields = [col for col in df.columns if pd.api.types.is_string_dtype(df[col])]
        if candidate_group_fields:
            group_field = candidate_group_fields[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"\nGrouped data by {group_field} (mean of {numeric_field}):")
            print(grouped_df.head())
    else:
        print("No numeric fields found for EDA.")
else:
    print("No DataFrame loaded to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_key and record_set_key in dataframes and numeric_fields:
    df = dataframes[record_set_key]
    field_to_plot = numeric_fields[0]
    plt.figure(figsize=(8, 4))
    sns.histplot(df[field_to_plot].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {field_to_plot} (Record set: {record_set_key})")
    plt.xlabel(field_to_plot)
    plt.ylabel("Count")
    plt.show()

    # If two numeric fields, scatter plot
    if len(numeric_fields) >= 2:
        plt.figure(figsize=(6, 6))
        sns.scatterplot(x=df[numeric_fields[0]], y=df[numeric_fields[1]])
        plt.xlabel(numeric_fields[0])
        plt.ylabel(numeric_fields[1])
        plt.title(f"{numeric_fields[0]} vs {numeric_fields[1]}")
        plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion
In this notebook, we've loaded and explored the Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells dataset using the `mlcroissant` library. We demonstrated loading metadata and records, reviewed available record sets and fields (using their `@id`s), performed exploratory analysis steps such as filtering records, normalizing, grouping, and visualizing the data.

For your own analyses, use the field and record set `@id` values shown in the overview steps, and alter the data processing to fit your research questions.
